In [2]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html, dash_table
from dash.dependencies import Input, Output
import plotly.express as px
import base64

JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import your CRUD Python module
from animal_shelter import AnimalShelter


###########################
# Data Manipulation / Model
###########################

username = "aacuser"
password = "CS340eband"

# Connect to database via CRUD module
db = AnimalShelter(username, password)

# Read all records from MongoDB
df = pd.DataFrame.from_records(db.read({}))

# Drop MongoDB _id column so Dash DataTable will not crash
if '_id' in df.columns:
    df.drop(columns=['_id'], inplace=True)


#########################
# Dashboard Layout / View
#########################

app = JupyterDash(__name__)

# Grazioso Salvare logo
image_filename = 'Grazioso Salvare Logo.png'
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

app.layout = html.Div([

    # Header
    html.Div([

        html.Div([
            html.Img(
                src='data:image/png;base64,{}'.format(encoded_image.decode()),
                style={'height': '90px', 'marginRight': '20px'}
            )
        ], style={
            'display': 'inline-block',
            'verticalAlign': 'middle'
        }),

        html.Div([
            html.H1('CS-340 Dashboard', style={
                'color': 'white',
                'margin': '0',
                'fontSize': '42px'
            }),
            html.H4('Project 2 Assignment', style={
                'color': 'white',
                'margin': '0'
            })
        ], style={
            'display': 'inline-block',
            'verticalAlign': 'middle'
        }),

        html.Div([
            dcc.Dropdown(
                id='filter-type',
                options=[
                    {'label': 'All/Reset', 'value': 'ALL'},
                    {'label': 'Water Rescue', 'value': 'WATER'},
                    {'label': 'Mountain/Wilderness Rescue', 'value': 'MOUNTAIN'},
                    {'label': 'Disaster/Tracking', 'value': 'DISASTER'}
                ],
                value='ALL',
                clearable=False,
                style={'width': '320px', 'color': 'black'}
            )
        ], style={
            'float': 'right',
            'marginTop': '20px'
        })

    ], style={
        'backgroundColor': '#5B6FA8',
        'padding': '20px',
        'borderRadius': '2px',
        'marginBottom': '15px',
        'display': 'flex',
        'alignItems': 'center',
        'justifyContent': 'space-between'
    }),

    # Unique identifier
    html.Div([
        html.H4('Ebony Anderson', style={
            'textAlign': 'center',
            'marginBottom': '10px',
            'margin': '5px 0 15px 0',
            'fontWeight': '500'
        })
    ]),

    html.Hr(),

    # Data table
    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i.replace('_', ' ').title(), "id": i} for i in df.columns],
        data=df.to_dict('records'),
        page_size=10,
        sort_action='native',
        filter_action='native',
        row_selectable='single',
        selected_rows=[0],
        style_table={
            'overflowX': 'auto',
            'marginBottom': '20px'
        },
        style_header={
            'backgroundColor': '#5B6FA8',
            'color': 'white',
            'fontWeight': 'bold',
            'textAlign': 'left'
        },
        style_cell={
            'textAlign': 'left',
            'padding': '8px',
            'fontFamily': 'Arial',
            'fontSize': '14px',
            'minWidth': '110px',
            'width': '110px',
            'maxWidth': '180px',
            'whiteSpace': 'normal'
        },
        style_data={
            'backgroundColor': '#F8F8F8',
            'color': '#222'
        }
    ),

    html.Hr(),

    # Pie chart and map side by side
    html.Div([
        html.Div(id='graph-id', style={
            'width': '48%',
            'display': 'inline-block',
            'verticalAlign': 'top',
            'paddingRight': '2%'
        }),
        html.Div(id='map-id', style={
            'width': '48%',
            'display': 'inline-block',
            'verticalAlign': 'top'
        })
    ], style={'marginBottom': '30px'}),

    # Bar chart below
    html.Div(id='bar-chart-id')

], style={
    'padding': '20px',
    'backgroundColor': '#EEF1F7'
})


#############################################
# Interaction Between Components / Controller
#############################################

@app.callback(
    Output('datatable-id', 'data'),
    [Input('filter-type', 'value')]
)
def update_dashboard(filter_type):

    if filter_type == 'WATER':
        query = {
            "animal_type": "Dog",
            "breed": {
                "$in": [
                    "Labrador Retriever Mix",
                    "Chesapeake Bay Retriever",
                    "Newfoundland"
                ]
            },
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }

    elif filter_type == 'MOUNTAIN':
        query = {
            "animal_type": "Dog",
            "breed": {
                "$in": [
                    "German Shepherd",
                    "Alaskan Malamute",
                    "Old English Sheepdog",
                    "Siberian Husky",
                    "Rottweiler"
                ]
            },
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }

    elif filter_type == 'DISASTER':
        query = {
            "animal_type": "Dog",
            "breed": {
                "$in": [
                    "Doberman Pinscher",
                    "German Shepherd",
                    "Golden Retriever",
                    "Bloodhound",
                    "Rottweiler"
                ]
            },
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300}
        }

    else:
        query = {}

    dff = pd.DataFrame.from_records(db.read(query))

    if '_id' in dff.columns:
        dff.drop(columns=['_id'], inplace=True)

    return dff.to_dict('records')


@app.callback(
    Output('graph-id', 'children'),
    [Input('datatable-id', 'derived_virtual_data')]
)
def update_graphs(viewData):

    if viewData is None or len(viewData) == 0:
        return []

    dff = pd.DataFrame.from_dict(viewData)

    breed_counts = dff['breed'].value_counts()
    top_breeds = breed_counts.nlargest(5)
    other_count = breed_counts.iloc[5:].sum()

    labels = list(top_breeds.index)
    values = list(top_breeds.values)

    if other_count > 0:
        labels.append('Other')
        values.append(other_count)

    fig = px.pie(
        names=labels,
        values=values,
        title='Breed Distribution',
        color_discrete_sequence=px.colors.sequential.Blues
    )

    fig.update_traces(
    textinfo='percent+label',
    pull=[0.05 if i == 0 else 0 for i in range(len(labels))],
    textfont_size=13
)
    
    fig.update_layout(
        height=450,
        paper_bgcolor='#F8F8F8',
        plot_bgcolor='#F8F8F8',
        font=dict(size=14),
        
        # cleaner spacing
        margin=dict(t=50, b=20, l=20, r=20),
        
        # better legend formatting
        legend=dict(
            font=dict(size=10),
            orientation="v"
        ),

        # center title nicely
        title_x=0.5
    )

    return [dcc.Graph(figure=fig)]


@app.callback(
    Output('bar-chart-id', 'children'),
    [Input('datatable-id', 'derived_virtual_data')]
)
def update_bar_chart(viewData):

    if viewData is None or len(viewData) == 0:
        return []

    dff = pd.DataFrame.from_dict(viewData)

    if 'sex_upon_outcome' not in dff.columns:
        return []

    sex_counts = dff['sex_upon_outcome'].value_counts().reset_index()
    sex_counts.columns = ['sex_upon_outcome', 'count']

    fig = px.bar(
        sex_counts,
        x='count',
        y='sex_upon_outcome',
        orientation='h',
        color='sex_upon_outcome',
        title='Breed Genders'
    )

    fig.update_layout(
        height=400,
        paper_bgcolor='#F8F8F8',
        plot_bgcolor='#E5ECF6',
        yaxis_title='Sex Upon Outcome',
        xaxis_title='Count',
        showlegend=True,
        
        yaxis=dict(categoryorder='total ascending'),
        title_x=0.5
        
    )

    return [dcc.Graph(figure=fig)]


@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    if selected_columns is None:
        return []

    return [{
        'if': {'column_id': i},
        'background_color': '#D2F3FF'
    } for i in selected_columns]


@app.callback(
    Output('map-id', 'children'),
    [Input('datatable-id', 'derived_virtual_data'),
     Input('datatable-id', 'derived_virtual_selected_rows')]
)
def update_map(viewData, index):

    if viewData is None or len(viewData) == 0:
        return []

    dff = pd.DataFrame.from_dict(viewData)

    if index is None or len(index) == 0:
        row = 0
    else:
        row = index[0]

    return [
        dl.Map(
            style={'width': '100%', 'height': '450px'},
            center=[30.75, -97.48],
            zoom=10,
            children=[
                dl.TileLayer(id="base-layer-id"),
                dl.Marker(
                    position=[dff.iloc[row]['location_lat'], dff.iloc[row]['location_long']],
                    children=[
                        dl.Tooltip(str(dff.iloc[row]['breed'])),
                        dl.Popup([
                            html.H4("Animal Name"),
                            html.P(str(dff.iloc[row]['name']))
                        ])
                    ]
                )
            ]
        )
    ]


# Run app and display result in JupyterLab mode
app.run_server()

Dash app running on https://aspecttoday-caramelgarlic-3000.codio.io/proxy/8050/
